# W4 Homework — A Guard Against Spinning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week04/W4_hw_loop_guard.ipynb)

**Goal.** Extend the lab's hand-built loop with a repetition guard — the stall
detector of notes Ch. 4 §4.6 — and show on a spinning run that the guard turns an
exhausted step budget into a finished answer. Then grow the mini evalset with two
multi-hop questions of your own.

Why it matters: the characteristic stall is the same Action with the same input,
turn after turn — a model that ignored the first identical result will ignore the
fifth; only the loop can notice.

The path: setup → the lab's loop, assembled → a run that spins → the guard ✍️ →
the evalset re-scored, plus your two questions ✍️ → completion.

*Runtime:* ~45 minutes. Due before the W5 session. Reference answers:
`labs/checkpoints/week04/solution.py`, published after the homework deadline.


## 1. Setup

*Do:* run the three cells; the last must print `ready`.


In [ ]:
%pip install -q "aisuite[openai,anthropic]"


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # alt: "anthropic:claude-haiku-4-5"


In [ ]:
import aisuite

client = aisuite.Client()


def chat(messages, **kwargs):
    """Messages -> assistant text."""
    response = client.chat.completions.create(model=MODEL, messages=messages, **kwargs)
    return response.choices[0].message.content


print(chat([{"role": "user", "content": "Reply with exactly: ready"}]))


## 2. The Loop, Assembled

Everything from the lab in one cell: the catalog, the tools, the ReAct protocol,
and the finished Judge (this is the published reference implementation — the
homework builds on the lab's completed state).

*Do:* run the cell; it re-checks the three judge test vectors.


In [ ]:
import json
import re

PAPER_CATALOG = {
    "react":            {"title": "ReAct", "authors": "Yao et al.", "year": 2022},
    "chain-of-thought": {"title": "Chain-of-Thought Prompting", "authors": "Wei et al.", "year": 2022},
    "toolformer":       {"title": "Toolformer", "authors": "Schick et al.", "year": 2023},
    "self-consistency": {"title": "Self-Consistency", "authors": "Wang et al.", "year": 2022},
}


def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not expression or not set(expression) <= allowed:
        return f"(calculator error: unsupported characters in {expression!r})"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"(calculator error: {exc})"


def paper_lookup(topic: str) -> str:
    entry = PAPER_CATALOG.get(topic.strip().lower())
    if entry is None:
        return (f"(unknown topic {topic!r}. Known topics: "
                + ", ".join(sorted(PAPER_CATALOG)) + ")")
    return f"{entry['title']} — {entry['authors']}, {entry['year']}"


TOOLS = {"calculator": calculator, "paper_lookup": paper_lookup}


def run_tool(name, tool_input):
    if name not in TOOLS:
        return f"(unregistered tool {name!r}. Available: " + ", ".join(sorted(TOOLS)) + ")"
    try:
        return TOOLS[name](tool_input)
    except Exception as exc:
        return f"(tool {name!r} raised: {exc})"


REACT_SYSTEM = """You are an agent that thinks and acts step by step. On every turn, answer in exactly one of the two formats below.

When a tool is needed:
Thought: <your reasoning so far>
Action: {"tool": "<tool name>", "input": "<input>"}

Available tools:
- calculator: evaluates an arithmetic expression (Python syntax, e.g. "2023 - 2022").
- paper_lookup: looks up a paper by topic key. Keys: react, chain-of-thought, toolformer, self-consistency.

When finalizing the answer:
Thought: <final reasoning>
Final Answer: <answer>

Observation: is filled in by the system — never write it yourself. Write nothing after Action."""


def judge(text):
    """Model output -> ("final", answer) | ("action", name, input) | ("error", msg)."""
    m = re.search(r"Final Answer:\s*(.*)", text, re.S)
    if m:
        return ("final", m.group(1).strip())
    m = re.search(r"Action:\s*(\{.*?\})", text, re.S)
    if m:
        try:
            call = json.loads(m.group(1))
        except json.JSONDecodeError as exc:
            return ("error", f"(Action JSON did not parse: {exc})")
        name, tool_input = call.get("tool"), call.get("input")
        if not isinstance(name, str) or not isinstance(tool_input, str):
            return ("error", '(Action JSON needs string fields "tool" and "input")')
        return ("action", name, tool_input)
    return ("error", "(output matched neither format)")


print(judge("Thought: done.\nFinal Answer: 42")[0],
      judge('Action: {"tool": "calculator", "input": "1+1"}')[0],
      judge("chatting")[0])


## 3. A Run That Spins

The question below invites re-verification, and some runs take the invitation
literally: the same lookup, again and again, each Observation identical. Without a
guard, the loop spends its whole budget and exits through the bound.

*Do:* run the cell and count in the trace how many times the identical Action ran.


In [ ]:
def agent_loop(question, system=REACT_SYSTEM, max_steps=6, verbose=True):
    """The lab's loop, unchanged: no guard."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    trace = []
    for step in range(1, max_steps + 1):
        output = chat(messages)
        trace.append(("model", output))
        if verbose:
            print(f"--- step {step} ---\n{output}")
        verdict = judge(output)
        if verdict[0] == "final":
            return verdict[1], trace
        if verdict[0] == "action":
            observation = run_tool(verdict[1], verdict[2])
        else:
            observation = verdict[1]
        trace.append(("observation", observation))
        if verbose:
            print(f"Observation: {observation}")
        messages.append({"role": "assistant", "content": output})
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    return f"(no Final Answer within max_steps={max_steps}; see trace)", trace


SPIN_QUESTION = ("In which year was ReAct published, according to the catalog? "
                 "Verify very carefully before answering.")

answer_spin, trace_spin = agent_loop(SPIN_QUESTION)
print("\nANSWER (no guard):", answer_spin)


## 4. The Guard ✍️

Write the guard into the loop copy below. Requirement: when the judged Action is
**identical to the previous executed action** (same tool, same input), do not
execute it again — reinject a guard notice instead, so the model must choose a
different action or finalize. The notice must (a) say the action was already run,
(b) point at the earlier result, and (c) name the two ways forward.

Hints — the ingredients:
1. Keep the last executed action in a variable: `last_action = None` before the
   loop; compare with `(name, tool_input)`.
2. The guard replaces only the Execute step — everything else (reinjection, the
   bound) stays as in Section 3.
3. Three lines of logic: `if (verdict[1], verdict[2]) == last_action: observation =
   "(guard: ...)"` / `else: observation = run_tool(...); last_action = ...`.


In [ ]:
def agent_loop_guarded(question, system=REACT_SYSTEM, max_steps=6, verbose=True):
    """The loop of Section 3 plus a repetition guard in the Execute step."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    trace = []
    last_action = None
    for step in range(1, max_steps + 1):
        output = chat(messages)
        trace.append(("model", output))
        if verbose:
            print(f"--- step {step} ---\n{output}")
        verdict = judge(output)
        if verdict[0] == "final":
            return verdict[1], trace
        if verdict[0] == "action":
            ### FILL IN (START) ###
            observation = run_tool(verdict[1], verdict[2])   # starter: no guard yet
            ### FILL IN (END) ###
        else:
            observation = verdict[1]
        trace.append(("observation", observation))
        if verbose:
            print(f"Observation: {observation}")
        messages.append({"role": "assistant", "content": output})
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    return f"(no Final Answer within max_steps={max_steps}; see trace)", trace


answer_guarded, trace_guarded = agent_loop_guarded(SPIN_QUESTION)
print("\nANSWER (guarded):", answer_guarded)
n_steps_guarded = sum(1 for kind, _ in trace_guarded if kind == "model")
print("model calls:", n_steps_guarded)


## 5. The Evalset, Re-scored — Plus Two of Your Own ✍️

The lab's five questions run through the guarded loop (target unchanged: ≥ 4/5).
Then add **two multi-hop questions of your own** over the same catalog and
calculator. Requirements: each needs at least two tool results to answer; keywords
name the checkable core of the answer; neither duplicates the five.

Hint: the multi-hop recipe from the lab — pick two catalog facts and one arithmetic
relation between them ("how many years", "which is earlier", "sum of the years").


In [ ]:
MINI_EVALSET = [
    {"question": "In which year was ReAct published, according to the catalog?",
     "keywords": ["2022"]},
    {"question": "Who are the authors of Toolformer, according to the catalog?",
     "keywords": ["schick"]},
    {"question": "Which was published earlier according to the catalog, Toolformer or chain-of-thought?",
     "keywords": ["chain-of-thought"]},
    {"question": "How many years passed between chain-of-thought and Toolformer, according to the catalog?",
     "keywords": ["1"]},
    {"question": "Who is the first author of self-consistency, and in which year was it published?",
     "keywords": ["wang", "2022"]},
]

### FILL IN (START) ###
MY_QUESTIONS = [
    {"question": "", "keywords": []},   # starter — write a multi-hop question
    {"question": "", "keywords": []},
]
### FILL IN (END) ###


def accuracy(evalset):
    correct = 0
    for item in evalset:
        if not item["question"]:
            continue
        answer, _ = agent_loop_guarded(item["question"], verbose=False)
        hit = all(k.lower() in str(answer).lower() for k in item["keywords"])
        correct += hit
        print(f"{'PASS' if hit else 'FAIL':4}  {item['question'][:58]}  -> {str(answer)[:48]}")
    return correct


core_score = accuracy(MINI_EVALSET)
print(f"\ncore: {core_score}/5")
mine_score = accuracy(MY_QUESTIONS)
print(f"mine: {mine_score}/2")


## 6. Completion Check

Submit: run the notebook top to bottom, then **File → Download → Download .ipynb**
and upload the file to the LMS.


In [ ]:
core_questions = {item["question"] for item in MINI_EVALSET}
completion = {
    "guarded run answers instead of exhausting the bound":
        "max_steps" not in str(answer_guarded),
    "guarded run used fewer steps than the bound":
        n_steps_guarded < 6,  # 6 = the loop's max_steps default
    "guard notice appears in the guarded trace":
        any("guard" in str(obs).lower() for kind, obs in trace_guarded
            if kind == "observation"),
    "core accuracy >= 4/5": core_score >= 4,
    "two own questions written, with keywords":
        all(q["question"] and q["keywords"] for q in MY_QUESTIONS),
    "own questions are new (not the core five)":
        all(q["question"] not in core_questions for q in MY_QUESTIONS),
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nHOMEWORK COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")
